# 19 — RS126 Enhanced + Kilitli ML Hibrit Deneyi

Bu notebook aşağıdaki beş portföyü aynı `2025+` ortak dönemde karşılaştırır:

```text
BIST100 Gross
Baseline Robot
RS126 Enhanced
Mevcut ML Challenger
RS126 Enhanced + mevcut ML filtresi
```

Hibrit akış:

```text
Baseline Robot skoru
→ RS126 <= 0 ise skor -1
→ Enhanced skor >= 11 ise aday
→ Kilitli aylık walk-forward Logistic Regression
→ Kilitli olasılık eşiği geçilirse AL
```

## Deneyde kilitli kalan kararlar

- ML hedefi
- Model ailesi
- Olasılık eşiği
- Embargo
- Aylık yeniden eğitim
- Strateji çıkışları
- Portföy risk kuralları

Hiçbir eşik yeniden seçilmez.

> **Metodolojik sınır:** RS126 Enhanced kuralı 2025+ sonuçları görüldükten sonra seçildi. Bu nedenle bu çalışma yeni ve bağımsız bir Audit değildir; tarihsel hibrit eleme testidir. Nihai kanıt, bundan sonraki ileri dönem paper-trading karşılaştırmasıdır.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import (
    add_robot_scores,
    build_market_regime,
)
from src.presets import (
    FINAL_PORTFOLIO_CONFIG,
    FINAL_STRATEGY_CONFIG,
)
from src.ml_dataset import (
    BASE_FEATURE_COLUMNS,
    add_meta_features,
)
from src.ml_targets import (
    add_alternative_targets,
)
from src.ml_walkforward import (
    WalkForwardConfig,
)
from src.rs126_ml_hybrid import (
    build_hybrid_probability_universes,
    evaluate_hybrid_experiment,
    load_locked_ml_spec,
    save_hybrid_artifacts,
)

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)

## 1. Kilitli ML kararını ve araştırma verilerini yükle

In [ ]:
locked_spec = load_locked_ml_spec(
    PROJECT_ROOT
)

display(
    pd.DataFrame(
        [
            {
                "Target": locked_spec.target,
                "Model": locked_spec.model_name,
                "Filter": locked_spec.filter_name,
                "Probability_Threshold": (
                    locked_spec.probability_threshold
                ),
                "Keep_Top_Fraction": (
                    locked_spec.keep_top_fraction
                ),
            }
        ]
    )
)

events = pd.read_parquet(
    PROJECT_ROOT
    / "results"
    / "ml"
    / "robot_meta_label_training.parquet"
)
events = add_alternative_targets(events)

for column in [
    "Signal_Date",
    "Entry_Date",
    "Exit_Date",
]:
    if column in events.columns:
        events[column] = pd.to_datetime(
            events[column]
        )

stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

print("Event sayısı:", len(events))
print("Hisse fiyat satırı:", len(stock_prices))
print("Endeks tarih aralığı:", market_prices["Date"].min(), "→", market_prices["Date"].max())

## 2. Baseline özelliklerini bir kez hazırla

ML özellikleri Baseline tanımlarıyla hesaplanır. RS126 kuralı daha sonra yalnızca aday uygunluğu ve portföy sıralamasını etkiler. Böylece kilitli ML modelinin `Score` ve `SCORE_RANK_PCT` anlamı değiştirilmez.

In [ ]:
stock_features = add_indicators(
    stock_prices
)
market_features = add_indicators(
    market_prices
)
market_regime = build_market_regime(
    market_features
)

baseline_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=False,
)

featured_baseline_prices = add_meta_features(
    scored_prices=baseline_prices,
    market_features=market_features,
)

print("Feature satırı:", len(featured_baseline_prices))
print(
    "Baseline AL sayısı:",
    int(
        featured_baseline_prices[
            "Signal"
        ].eq("AL").sum()
    ),
)

## 3. Aylık walk-forward olasılıkları üret

Model her ay yalnızca geçmişte tamamlanmış işlemlerle yeniden eğitilir. RS126 Enhanced adayları Baseline AL kümesinin alt kümesi olduğu için model bir kez çalıştırılır; aynı `Date + Ticker` olasılığı hibrit portföyde yeniden kullanılır.

In [ ]:
EXPERIMENT_START = "2025-01-01"
EXPERIMENT_END = min(
    pd.to_datetime(
        featured_baseline_prices["Date"]
    ).max(),
    pd.to_datetime(
        market_prices["Date"]
    ).max(),
).strftime("%Y-%m-%d")

walk_forward_config = WalkForwardConfig(
    start=EXPERIMENT_START,
    end=EXPERIMENT_END,
    retrain_frequency="MS",
    embargo_days=5,
    minimum_training_events=500,
    random_state=42,
)

(
    baseline_probability_prices,
    enhanced_probability_prices,
    training_log,
) = build_hybrid_probability_universes(
    featured_baseline_prices=(
        featured_baseline_prices
    ),
    events=events,
    market_features=market_features,
    spec=locked_spec,
    walk_forward_config=(
        walk_forward_config
    ),
    strategy_config=(
        FINAL_STRATEGY_CONFIG
    ),
    feature_columns=BASE_FEATURE_COLUMNS,
)

display(
    training_log[
        [
            "Block_Start",
            "Block_End",
            "Training_Event_Count",
            "Robot_AL_Rows",
            "Scored_AL_Rows",
            "Score_Coverage_%",
            "Rows_With_Imputation_%",
        ]
    ]
)

print(
    "Walk-forward tarih aralığı:",
    EXPERIMENT_START,
    "→",
    EXPERIMENT_END,
)

## 4. Dört strateji ve BIST100 karşılaştırması

In [ ]:
artifacts = evaluate_hybrid_experiment(
    baseline_probability_prices=(
        baseline_probability_prices
    ),
    enhanced_probability_prices=(
        enhanced_probability_prices
    ),
    market_prices=market_prices,
    spec=locked_spec,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    start=EXPERIMENT_START,
    end=EXPERIMENT_END,
    training_log=training_log,
)

metric_columns = [
    "Portfolio",
    "Candidate_Universe",
    "ML_Applied",
    "End_Value",
    "Total_Return_%",
    "CAGR_%",
    "Max_Drawdown_%",
    "Profit_Factor",
    "Sharpe",
    "Sortino",
    "Calmar",
    "Trade_Count",
    "Exposure_%",
    "Average_Open_Positions",
    "Signal_Pass_Rate_%",
]

display(
    artifacts.metrics[
        [
            column
            for column in metric_columns
            if column in artifacts.metrics.columns
        ]
    ]
)

## 5. Hibritin referanslara göre katkısı

Ana kıyas:

```text
RS126 Enhanced + ML
vs
RS126 Enhanced
```

Bu karşılaştırma ML katmanının Enhanced aday evrenine ek katkısını ölçer.

In [ ]:
display(
    artifacts.pairwise
)

display(
    artifacts.acceptance
)

### Tarihsel eleme kriterleri

**Getiri odaklı geçiş**

```text
CAGR farkı               >= +1,5 yüzde puan
Drawdown                 kötüleşmemeli
Calmar iyileşmesi        >= %5
Profit Factor            düşmemeli
İşlem sayısı             RS126 Enhanced'ın en az %70'i
```

**Risk odaklı geçiş**

```text
CAGR farkı               >= -1,0 yüzde puan
Drawdown iyileşmesi      >= 2 yüzde puan
Calmar iyileşmesi        >= %8
Profit Factor oranı      >= %95
İşlem sayısı             RS126 Enhanced'ın en az %70'i
```

Bu kriterleri geçmesi, yalnızca ileri dönem paper-trading adaylığı anlamına gelir.

## 6. Aday hunisi ve aşırı filtreleme kontrolü

In [ ]:
display(
    artifacts.funnel
)

hybrid_row = artifacts.metrics.loc[
    artifacts.metrics["Portfolio"].eq(
        "RS126_Enhanced_ML"
    )
].iloc[0]

enhanced_row = artifacts.metrics.loc[
    artifacts.metrics["Portfolio"].eq(
        "RS126_Enhanced"
    )
].iloc[0]

print(
    "Hibrit işlem oranı / Enhanced:",
    round(
        hybrid_row["Trade_Count"]
        / enhanced_row["Trade_Count"],
        4,
    ),
)
print(
    "Hibrit exposure farkı:",
    round(
        hybrid_row["Exposure_%"]
        - enhanced_row["Exposure_%"],
        2,
    ),
    "puan",
)

## 7. Equity grafiği

In [ ]:
equity_columns = [
    "BIST100_Gross",
    "Baseline_Robot",
    "RS126_Enhanced",
    "ML_Challenger",
    "RS126_Enhanced_ML",
]

plt.figure(figsize=(15, 8))

for column in equity_columns:
    if column in artifacts.equity.columns:
        plt.plot(
            artifacts.equity["Date"],
            artifacts.equity[column],
            label=column,
        )

plt.title(
    "Ortak 2025+ Dönem — RS126 Enhanced + ML Hibrit Karşılaştırması"
)
plt.xlabel("Tarih")
plt.ylabel("Portföy Değeri (TL)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Yıllık getiri ve aktif performans

In [ ]:
display(
    artifacts.yearly
)

display(
    artifacts.active.sort_values(
        "Portfolio"
    )
)

## 9. Sonuçları kaydet

Çıktılar `.gitignore` kapsamındaki yerel klasöre yazılır.

In [ ]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "ml"
    / "rs126_hybrid"
)

saved_paths = save_hybrid_artifacts(
    output_directory=OUTPUT_DIR,
    artifacts=artifacts,
)

for name, path in saved_paths.items():
    print(name, "→", path)

## Karar çerçevesi

### Tarihsel eleme başarısızsa

```text
RS126 Enhanced korunur
Hibrit üretime veya uygulamaya eklenmez
```

### Tarihsel eleme başarılıysa

```text
RS126 Enhanced + ML
→ Yeni bağımsız paper-trading Challenger adayı
→ Mevcut üç üretim portföyü değiştirilmez
→ İleri dönem performansı beklenir
```

Bu notebook hiçbir FastAPI, Streamlit veya günlük sinyal dosyasını değiştirmez.